In [26]:
import os
import pandas as pd
import pandas as pd
import numpy as np
import requests
import time

from pathlib import Path
from getpass import getpass

movies = pd.read_csv("D:\\Programming\\DI-Bootcamp\\hack1_movie_rec\\data\\raw\\movies.csv")
ratings = pd.read_csv("D:\\Programming\\DI-Bootcamp\\hack1_movie_rec\\data\\raw\\ratings.csv")
links = pd.read_csv("D:\\Programming\\DI-Bootcamp\\hack1_movie_rec\\data\\raw\\links.csv")
tags = pd.read_csv("D:\\Programming\\DI-Bootcamp\\hack1_movie_rec\\data\\raw\\tags.csv")

In [7]:
print(movies.shape)
print(ratings.shape)

print(movies.head())
print(ratings.head())

(9742, 3)
(100836, 4)
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931


In [8]:
movie_ratings = ratings.merge(
    movies,
    on="movieId",
    how="left"
)

In [9]:
movie_ratings["watch_date"] = pd.to_datetime(
    movie_ratings["timestamp"],
    unit="s"
)

In [11]:
print(movie_ratings.head())

   userId  movieId  rating  timestamp                        title  \
0       1        1     4.0  964982703             Toy Story (1995)   
1       1        3     4.0  964981247      Grumpier Old Men (1995)   
2       1        6     4.0  964982224                  Heat (1995)   
3       1       47     5.0  964983815  Seven (a.k.a. Se7en) (1995)   
4       1       50     5.0  964982931   Usual Suspects, The (1995)   

                                        genres          watch_date  
0  Adventure|Animation|Children|Comedy|Fantasy 2000-07-30 18:45:03  
1                               Comedy|Romance 2000-07-30 18:20:47  
2                        Action|Crime|Thriller 2000-07-30 18:37:04  
3                             Mystery|Thriller 2000-07-30 19:03:35  
4                       Crime|Mystery|Thriller 2000-07-30 18:48:51  


In [16]:
movie_ratings = movie_ratings.merge(
    links[["movieId", "tmdbId"]],
    on="movieId",
    how="left"
)

In [17]:
movie_ratings["tmdbId"] = pd.to_numeric(
    movie_ratings["tmdbId"],
    errors="coerce"
).astype("Int64")

In [18]:
movie_ratings[
    ["userId", "movieId", "title", "rating", "tmdbId"]
].head()

,userId,movieId,title,rating,tmdbId
0,1,1,Toy Story (1995),4.0,862
1,1,3,Grumpier Old Men (1995),4.0,15602
2,1,6,Heat (1995),4.0,949
3,1,47,Seven (a.k.a. Se7en) (1995),5.0,807
4,1,50,"Usual Suspects, The (1995)",5.0,629


In [19]:
unique_tmdb_ids = (
    movie_ratings["tmdbId"]
    .dropna()
    .astype(int)
    .unique()
)

print(f"Movie rating rows: {len(movie_ratings):,}")
print(f"Unique TMDB movies: {len(unique_tmdb_ids):,}")

Movie rating rows: 100,836
Unique TMDB movies: 9,715


In [21]:
TMDB_CACHE = Path(
    "D:\\Programming\\DI-Bootcamp\\hack1_movie_rec\\data\\processed\\tmdb_movies.csv"
)

TMDB_CACHE.parent.mkdir(
    parents=True,
    exist_ok=True
)

print(f"TMDB cache location: {TMDB_CACHE}")

TMDB cache location: D:\Programming\DI-Bootcamp\hack1_movie_rec\data\processed\tmdb_movies.csv


In [22]:
if TMDB_CACHE.exists():
    
    tmdb_movies = pd.read_csv(
        TMDB_CACHE
    )
    
    print(
        f"Cached TMDB data found: "
        f"{len(tmdb_movies):,} movies"
    )

else:
    
    tmdb_movies = pd.DataFrame()
    
    print("No cached TMDB data found yet.")

No cached TMDB data found yet.


In [23]:
if not tmdb_movies.empty:

    tmdb_movies["tmdbId"] = pd.to_numeric(
        tmdb_movies["tmdbId"],
        errors="coerce"
    )

    if "tmdb_status" in tmdb_movies.columns:

        completed_ids = set(
            tmdb_movies.loc[
                tmdb_movies["tmdb_status"].isin(
                    ["ok", "not_found"]
                ),
                "tmdbId"
            ]
            .dropna()
            .astype(int)
        )

    else:

        completed_ids = set(
            tmdb_movies["tmdbId"]
            .dropna()
            .astype(int)
        )

else:

    completed_ids = set()


remaining_ids = [
    tmdb_id
    for tmdb_id in unique_tmdb_ids
    if tmdb_id not in completed_ids
]


print(f"Already completed: {len(completed_ids):,}")
print(f"Still to download: {len(remaining_ids):,}")

Already completed: 0
Still to download: 9,715


In [24]:
def get_tmdb_movie(tmdb_id, session, max_retries=3):
    """
    Download metadata for one movie from TMDB.
    
    Returns movie information including:
    title, director, genres, release date, runtime,
    language, rating, popularity and overview.
    """

    url = (
        f"https://api.themoviedb.org/3/movie/"
        f"{int(tmdb_id)}"
    )

    params = {
        "language": "en-US",
        "append_to_response": "credits"
    }

    for attempt in range(1, max_retries + 1):

        try:

            response = session.get(
                url,
                params=params,
                timeout=30
            )

            # TMDB rate limit
            if response.status_code == 429:

                wait_time = int(
                    response.headers.get(
                        "Retry-After",
                        2
                    )
                )

                print(
                    f"Rate limit reached. "
                    f"Waiting {wait_time} seconds..."
                )

                time.sleep(wait_time)

                continue

            # Movie no longer exists / unavailable
            if response.status_code == 404:

                return {
                    "tmdbId": int(tmdb_id),
                    "tmdb_status": "not_found"
                }

            response.raise_for_status()

            data = response.json()

            # -----------------------------
            # Director
            # -----------------------------

            crew = (
                data
                .get("credits", {})
                .get("crew", [])
            )

            directors = [
                person["name"]
                for person in crew
                if person.get("job") == "Director"
                and person.get("name")
            ]

            # -----------------------------
            # Genres
            # -----------------------------

            tmdb_genres = [
                genre["name"]
                for genre in data.get(
                    "genres",
                    []
                )
                if genre.get("name")
            ]

            # -----------------------------
            # Return selected data
            # -----------------------------

            return {
                "tmdbId": int(tmdb_id),
                "tmdb_status": "ok",

                "tmdb_title":
                    data.get("title"),

                "original_title":
                    data.get("original_title"),

                "director":
                    " | ".join(directors),

                "tmdb_genres":
                    " | ".join(tmdb_genres),

                "release_date":
                    data.get("release_date"),

                "runtime":
                    data.get("runtime"),

                "original_language":
                    data.get("original_language"),

                "tmdb_rating":
                    data.get("vote_average"),

                "tmdb_vote_count":
                    data.get("vote_count"),

                "popularity":
                    data.get("popularity"),

                "overview":
                    data.get("overview")
            }

        except requests.RequestException as error:

            print(
                f"Attempt {attempt}/{max_retries} "
                f"failed for TMDB ID {tmdb_id}: "
                f"{error}"
            )

            if attempt < max_retries:
                time.sleep(2)

    return {
        "tmdbId": int(tmdb_id),
        "tmdb_status": "error"
    }

In [27]:
TMDB_TOKEN = None

if len(remaining_ids) > 0:

    TMDB_TOKEN = os.getenv(
        "TMDB_TOKEN"
    )

    if not TMDB_TOKEN:

        TMDB_TOKEN = getpass(
            "Enter your TMDB API Read Access Token: "
        )

    print("TMDB authentication ready.")

else:

    print(
        "All TMDB data is already cached. "
        "No API token is needed."
    )

TMDB authentication ready.


In [29]:
# Test the TMDB connection with one movie

if len(remaining_ids) > 0:

    test_session = requests.Session()

    test_session.headers.update({
        "Authorization": f"Bearer {TMDB_TOKEN}",
        "accept": "application/json"
    })

    test_tmdb_id = remaining_ids[0]

    print(f"Testing TMDB ID: {test_tmdb_id}")

    test_movie = get_tmdb_movie(
        test_tmdb_id,
        test_session
    )

    test_session.close()

    print("\nTMDB response:")
    display(test_movie)

else:

    print(
        "No download test needed because "
        "all TMDB movies are already cached."
    )

Testing TMDB ID: 862

TMDB response:


{'tmdbId': 862,
 'tmdb_status': 'ok',
 'tmdb_title': 'Toy Story',
 'original_title': 'Toy Story',
 'director': 'John Lasseter',
 'tmdb_genres': 'Family | Comedy | Animation | Adventure',
 'release_date': '1995-11-22',
 'runtime': 81,
 'original_language': 'en',
 'tmdb_rating': 8.0,
 'tmdb_vote_count': 20304,
 'popularity': 41.0572,
 'overview': "Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences."}

In [30]:
if len(remaining_ids) > 0:

    session = requests.Session()

    session.headers.update({
        "Authorization": f"Bearer {TMDB_TOKEN}",
        "accept": "application/json"
    })

    newly_downloaded = []

    for i, tmdb_id in enumerate(
        remaining_ids,
        start=1
    ):

        movie_data = get_tmdb_movie(
            tmdb_id,
            session
        )

        newly_downloaded.append(
            movie_data
        )


        # Save every 100 movies
        if i % 100 == 0:

            new_df = pd.DataFrame(
                newly_downloaded
            )

            tmdb_movies = pd.concat(
                [
                    tmdb_movies,
                    new_df
                ],
                ignore_index=True
            )

            tmdb_movies = (
                tmdb_movies
                .drop_duplicates(
                    subset="tmdbId",
                    keep="last"
                )
            )

            tmdb_movies.to_csv(
                TMDB_CACHE,
                index=False
            )

            newly_downloaded = []

            print(
                f"Downloaded {i:,} / "
                f"{len(remaining_ids):,}"
            )


    # Add whatever remains after the last batch of 100
    if newly_downloaded:

        new_df = pd.DataFrame(
            newly_downloaded
        )

        tmdb_movies = pd.concat(
            [
                tmdb_movies,
                new_df
            ],
            ignore_index=True
        )


    tmdb_movies = (
        tmdb_movies
        .drop_duplicates(
            subset="tmdbId",
            keep="last"
        )
    )


    tmdb_movies.to_csv(
        TMDB_CACHE,
        index=False
    )

    session.close()

    print("\nDownload finished.")

else:

    print(
        "TMDB dataset already complete. "
        "No download necessary."
    )

Downloaded 100 / 9,715
Downloaded 200 / 9,715
Downloaded 300 / 9,715
Downloaded 400 / 9,715
Downloaded 500 / 9,715
Downloaded 600 / 9,715
Downloaded 700 / 9,715
Downloaded 800 / 9,715
Downloaded 900 / 9,715
Downloaded 1,000 / 9,715
Downloaded 1,100 / 9,715
Downloaded 1,200 / 9,715
Downloaded 1,300 / 9,715
Downloaded 1,400 / 9,715
Downloaded 1,500 / 9,715
Downloaded 1,600 / 9,715
Downloaded 1,700 / 9,715
Downloaded 1,800 / 9,715
Downloaded 1,900 / 9,715
Downloaded 2,000 / 9,715
Downloaded 2,100 / 9,715
Downloaded 2,200 / 9,715
Downloaded 2,300 / 9,715
Downloaded 2,400 / 9,715
Downloaded 2,500 / 9,715
Downloaded 2,600 / 9,715
Downloaded 2,700 / 9,715
Downloaded 2,800 / 9,715
Downloaded 2,900 / 9,715
Downloaded 3,000 / 9,715
Downloaded 3,100 / 9,715
Downloaded 3,200 / 9,715
Downloaded 3,300 / 9,715
Downloaded 3,400 / 9,715
Downloaded 3,500 / 9,715
Downloaded 3,600 / 9,715
Downloaded 3,700 / 9,715
Downloaded 3,800 / 9,715
Downloaded 3,900 / 9,715
Downloaded 4,000 / 9,715
Downloaded 4,100 /